# Definizioni iniziali

### Pacchetti

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

### Matplotlib

In [ ]:
def show_image(img, title=None, cmap=None):
    cmap = cmap or ('gray' if len(img.shape) == 2 else None)
    img = img
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    figure, axes = plt.subplots(figsize=(20, 20))
    axes.imshow(img, cmap=cmap)
    axes.set_title(title)
    axes.axis('off')
    plt.show()

def show_horizontal_images(images, cmap=None):
    figure, subplots = plt.subplots(1, len(images), figsize=(20, 20))
    figure.subplots_adjust(wspace=0.01)
    for i, image in enumerate(images):
        if len(image) == 3:
            img, title, _cmap = image
        else:
            img, title = image
            _cmap = cmap or ('gray' if len(img.shape) == 2 else None)

        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        subplots[i].imshow(img, cmap=_cmap)
        subplots[i].set_title(title)
        subplots[i].axis('off')
    plt.show()

# Maschera

In [ ]:
label_free = cv2.imread("../../Materiale/Prove/non_colorato.png")

# Threshold or binarize the image (if needed)
_, binary = cv2.threshold(cv2.cvtColor(label_free, cv2.COLOR_BGR2GRAY), 230, 255, cv2.THRESH_BINARY)
show_horizontal_images([(label_free, "Non colorata"), (binary, "Binaria")])

# Find contours or connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)
print(f"Number of components: {num_labels}")

# Sort by area
sorted_indices = np.argsort(stats[:, cv2.CC_STAT_AREA])[::-1]
filtered_labels = np.zeros_like(labels)
for i in sorted_indices[1:10]:
    filtered_labels[labels == i] = i
show_horizontal_images([(labels, "Etichette"), (filtered_labels, "Etichette filtrate")], cmap='nipy_spectral')

# Create an empty mask to store only "good" components
mask_label_free = np.zeros_like(binary)

# Loop through each component (skip background at index 0)
for i in sorted_indices[1:10]:
    x, y, w, h, area = stats[i]

    # Extract the region of interest
    component_mask = (labels == i).astype(np.uint8) * 255
    roi = label_free[y:y+h, x:x+w]
    roi_mask = component_mask[y:y+h, x:x+w]

    # Calculate standard deviation within the ROI using the mask
    std_dev = np.std(roi[roi_mask == 255])
    print(i, std_dev)

    # Filter based on standard deviation (and optionally area)
    if std_dev > 5:  # Change threshold based on your needs
        mask_label_free[labels == i] = 255

# Applica la maschera all'immagine
mask_label_free = cv2.bitwise_not(mask_label_free)
masked_label_free = cv2.bitwise_and(label_free, label_free, mask=mask_label_free)

# Mostra le immagini
show_horizontal_images([
    (label_free, "Non colorata"),
    (mask_label_free, "Maschera"),
    (masked_label_free, "Non colorata con maschera")])

In [ ]:
images = []
for i in sorted_indices[1:10]:
    filtered_labels = np.zeros_like(labels)
    filtered_labels[labels == i] = 255
    images.append((filtered_labels, f"Etichetta {i}"))
show_horizontal_images(images, cmap='nipy_spectral')